In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys

from pathlib import Path
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.base import clone

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import build_features

df = pd.read_csv(
    DATA_DIR / "train-test.csv",
    parse_dates=["date"]
)

cutoff_date = pd.Timestamp("2025-10-01")

train_raw = df[df["date"] < cutoff_date].copy()
holdout_raw = df[df["date"] >= cutoff_date].copy()

X_train = build_features(train_raw)
y_train = train_raw["posted_rate"].copy()

X_holdout = build_features(holdout_raw)
y_holdout = holdout_raw["posted_rate"].copy()

print("Training:", X_train.shape)
print("Holdout:", X_holdout.shape)


Training: (43147, 12)
Holdout: (4853, 12)


In [2]:

from sklearn.dummy import DummyRegressor

baseline = DummyRegressor(
    strategy="median"
)

baseline.fit(X_train, y_train)

baseline_predictions = baseline.predict(
    X_holdout
)

baseline_mae = mean_absolute_error(
    y_holdout,
    baseline_predictions
)

print("Predicted constant rate:",
      baseline.constant_[0, 0])

print("Baseline MAE: $",
      round(baseline_mae, 2))


Predicted constant rate: 2029.7
Baseline MAE: $ 1146.79


In [3]:

# Calculate training rate per mile
train_rpm = (
    y_train / X_train["distance"]
)

# Build training-only lookup table
rpm_table = pd.DataFrame({
    "equipment": X_train["equipment"],
    "rpm": train_rpm
})

equipment_rpm = (
    rpm_table.groupby("equipment")["rpm"]
    .median()
)

print(equipment_rpm)

# Fallback for unseen equipment categories
global_rpm = train_rpm.median()

# Predict October rates
holdout_rpm = (
    X_holdout["equipment"]
    .map(equipment_rpm)
    .fillna(global_rpm)
)

rpm_predictions = (
    X_holdout["distance"] * holdout_rpm
).to_numpy()

rpm_mae = mean_absolute_error(
    y_holdout,
    rpm_predictions
)

print("Equipment RPM baseline MAE: $",
      round(rpm_mae, 2))


equipment
Dry Van    2.043653
Flatbed    2.219193
Reefer     2.311748
Name: rpm, dtype: float64
Equipment RPM baseline MAE: $ 231.3


In [4]:

categorical_cols = [
    "pickup",
    "delivery",
    "equipment"
]

numerical_cols = [
    col for col in X_train.columns
    if col not in categorical_cols
]

numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "encoder",
        OneHotEncoder(handle_unknown="ignore")
    )
])

preprocessor = ColumnTransformer([
    ("numerical", numerical_pipeline, numerical_cols),
    ("categorical", categorical_pipeline, categorical_cols)
])


In [5]:

from lightgbm import LGBMRegressor

lgbm = LGBMRegressor(
    objective="regression_l1",
    n_estimators=350,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=50,
    reg_lambda=3.0,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model_pipeline = Pipeline([
    ("preprocessing", clone(preprocessor)),
    ("model", lgbm)
])

model_pipeline.fit(X_train, y_train)

lgbm_predictions = model_pipeline.predict(
    X_holdout
)

print("LightGBM training completed!")

print("First 5 predictions:")
print(lgbm_predictions[:5])


LightGBM training completed!
First 5 predictions:
[4068.64865757 2493.3081227  2568.46499161 1537.87724161 3127.80601188]


/Users/ahmedali/spotter-freight-rate-prediction/.venv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [6]:

def evaluate_model(name, actual, predicted):
    mae = mean_absolute_error(actual, predicted)

    rmse = np.sqrt(
        mean_squared_error(actual, predicted)
    )

    wape = (
        np.abs(actual.to_numpy() - predicted).sum()
        / np.abs(actual.to_numpy()).sum()
    ) * 100

    r2 = r2_score(actual, predicted)

    return {
        "Model": name,
        "MAE": round(mae, 2),
        "RMSE": round(rmse, 2),
        "WAPE (%)": round(wape, 2),
        "R2": round(r2, 4)
    }


results = pd.DataFrame([
    evaluate_model(
        "Median Baseline",
        y_holdout,
        baseline_predictions
    ),
    evaluate_model(
        "Equipment RPM Baseline",
        y_holdout,
        rpm_predictions
    ),
    evaluate_model(
        "LightGBM",
        y_holdout,
        lgbm_predictions
    )
])

results = results.sort_values("MAE")

display(results)


,Model,MAE,RMSE,WAPE (%),R2
2,LightGBM,112.70,647.35,4.74,0.8206
1,Equipment RPM Baseline,231.30,683.87,9.72,0.7998
0,Median Baseline,1146.79,1567.97,48.20,-0.0522


In [7]:

error_analysis = holdout_raw[
    ["load_id", "pickup", "delivery",
     "distance", "equipment", "posted_rate"]
].copy()

error_analysis["predicted_rate"] = (
    lgbm_predictions
)

error_analysis["absolute_error"] = (
    error_analysis["posted_rate"]
    - error_analysis["predicted_rate"]
).abs()

display(
    error_analysis
    .sort_values("absolute_error", ascending=False)
    .head(10)
)


,load_id,pickup,delivery,distance,equipment,posted_rate,predicted_rate,absolute_error
43539,TR-043540,Milwaukee,Bakersfield,1893.0,Dry Van,17514.11,3645.171430,13868.938570
46213,TR-046214,Columbia,Tucson,2023.4,Flatbed,17893.17,4225.552327,13667.617673
47298,TR-047299,Boston,Bakersfield,2979.1,Reefer,19110.30,6002.200131,13108.099869
46516,TR-046517,Toledo,Albuquerque,1684.3,Reefer,14784.38,3739.763740,11044.616260
46366,TR-046367,Bakersfield,Columbia,2251.9,Dry Van,14403.14,4358.097267,10045.042733
47536,TR-047537,Los Angeles,Richmond,2782.4,Dry Van,15003.55,4969.512774,10034.037226
44267,TR-044268,Albany,Albuquerque,2492.6,Dry Van,14425.42,4456.840193,9968.579807
43384,TR-043385,Lexington,Lubbock,1187.0,Dry Van,11696.44,2383.752224,9312.687776
43302,TR-043303,Atlanta,Phoenix,2083.1,Reefer,13894.55,4646.036754,9248.513246
47586,TR-047587,Corpus Christi,Las Vegas,1383.3,Flatbed,11596.28,2986.435938,8609.844062


In [8]:

equipment_errors = (
    error_analysis.groupby("equipment")
    .agg(
        loads=("absolute_error", "count"),
        MAE=("absolute_error", "mean"),
        median_error=("absolute_error", "median")
    )
    .round(2)
)

display(equipment_errors)


,loads,MAE,median_error
equipment,,,
Dry Van,2745,94.51,24.80
Flatbed,918,143.77,76.43
Reefer,1190,130.67,44.24
